In [1]:
from tomllib import load as toml_load

In [2]:
# do toml things.
config_file = "./prefigure/tests/test-prefigure.toml"
with open(config_file, "rb") as toml_file:
    config = toml_load(toml_file)

with open(config_file, "r") as toml_file:
    config_text = toml_file.readlines()

config, config_text

({'logging': {'name': {'arg_options': {'default': 'Example Run',
     'help': 'Name of the run. Used for logging.',
     'flags': ['--name', '-n']}},
   'project': {'arg_options': {'default': 'Example Project',
     'help': 'Name of the project. Used for logging.',
     'flags': ['--project', '-p']}},
   'wandb': {'run_id': {'arg_options': {'default': 'Example Run',
      'help': 'You can specify a run-id if you would like wandb to continue logging from that checkpoint.',
      'flags': ['--project', '-p']}},
    'ckpt_name': {'arg_options': {'default': 'example_ckpt'}}}},
  'training': {'seed': 42,
   'batch_size': 256,
   'precision': '16-mixed',
   'strategy': 'auto',
   'save_dir': '/home/eduardo/Projects/pre_encode_audio/data/hyperencoder_training'},
  'validation': {'val_every': -1},
  'checkpointing': {'save_top_k': 10, 'checkpoint_every': 1},
  'dataloaders': {'num_workers': 8, 'persistent_workers': False},
  'recovery': {'recover_latest': False, 'ckpt_path': ''},
  'pretrained

In [3]:
argument_parser_objs = {
    'prog',
    'usage',
    'description',
    'epilog',
    'prefix_chars',
    'argument_default',
    'add_help',
    'allow_abbrev',
    'exit_on_error'
}
escape_key = 'escape_argument'
arg_key = 'arg_options'

add_argument_args = {
    "name": lambda val: [val],
    "flags": lambda val: val,
}

add_argument_kwargs = {
    "action",
    "nargs",
    "const",
    "default",
    "type",
    "choices",
    "required",
    "help",
    "metavar",
    "dest",
    "deprecated"
}

def extract_args_options(option_dict):
    args_dict = {
        'args': [],
        'kwargs': {}
    }

    for key, item in option_dict.items():
        if key in add_argument_args:
            args_dict['args'] = add_argument_args[key](item)
        elif key in add_argument_kwargs:
            args_dict['kwargs'][key] = item

    return args_dict

def traverse_config(config):
    config_dict = {}

    if isinstance(config, dict):
        # print('Entry Point found Dict with keys:', list(config.keys()))
        for key, item in config.items():
            if isinstance(item, dict):
                if key == arg_key:
                    # print('Found Arg Options:', key)
                    return extract_args_options(item)
                else:
                    # print('Found Key mapping to Dict:', key)
                    ret_items = traverse_config(item)
                    config_dict.update({key: ret_items})
            else:
                # print('Found Key Mapping to Item, Key:', key)
                config_dict[key] = {'args': [], 'kwargs': {'default': item}}

        return config_dict
    else:
        # print('Entry Point found Item', config)
        return config

t_config = traverse_config(config)

t_config

{'logging': {'name': {'args': ['--name', '-n'],
   'kwargs': {'default': 'Example Run',
    'help': 'Name of the run. Used for logging.'}},
  'project': {'args': ['--project', '-p'],
   'kwargs': {'default': 'Example Project',
    'help': 'Name of the project. Used for logging.'}},
  'wandb': {'run_id': {'args': ['--project', '-p'],
    'kwargs': {'default': 'Example Run',
     'help': 'You can specify a run-id if you would like wandb to continue logging from that checkpoint.'}},
   'ckpt_name': {'args': [], 'kwargs': {'default': 'example_ckpt'}}}},
 'training': {'seed': {'args': [], 'kwargs': {'default': 42}},
  'batch_size': {'args': [], 'kwargs': {'default': 256}},
  'precision': {'args': [], 'kwargs': {'default': '16-mixed'}},
  'strategy': {'args': [], 'kwargs': {'default': 'auto'}},
  'save_dir': {'args': [],
   'kwargs': {'default': '/home/eduardo/Projects/pre_encode_audio/data/hyperencoder_training'}}},
 'validation': {'val_every': {'args': [], 'kwargs': {'default': -1}}},
 'ch

In [4]:
from collections import defaultdict
line_counter = 0
curr_key_path = []
comment_accum = []
last_comment_line = -1
update_dict = defaultdict(dict)
for line in config_text:
    c_line = line.strip()
    # print('Key Path', curr_key_path)
    if len(c_line) > 0:
        match c_line[0]:
            case '[':
                # print('Found Key:', c_line)
                curr_key_path = c_line[1:len(c_line) - 1].split('.')
            case '#':
                # print('Found Comment:', c_line)
                comment_accum.append(c_line.split("#",1)[-1].strip())
                last_comment_line = line_counter
            case _:
                if len(comment_accum) > 0:
                    if line_counter - last_comment_line == 1:
                        k = c_line.split('=')[0].strip()
                        # print(f'Key: {k} found, and comment in last line:', c_line)
                        d = update_dict
                        for k_v in curr_key_path:
                            d = d[k_v]
                        d[k] = {
                            # 'args': [],
                            'kwargs': {
                                'help': ' '.join(comment_accum)
                            }
                        }
                        
                    last_comment_line = -1
                    comment_accum = []
                        
    line_counter += 1
update_dict

defaultdict(dict,
            {'training': {'seed': {'kwargs': {'help': 'The random seed'}},
              'batch_size': {'kwargs': {'help': 'The batch size'}},
              'precision': {'kwargs': {'help': 'Precision to use for training'}},
              'strategy': {'kwargs': {'help': 'Multi-GPU strategy for PyTorch Lightning, should be a #'}},
              'save_dir': {'kwargs': {'help': 'Directory to save checkpoints and assorted model files in'}}},
             'validation': {'val_every': {'kwargs': {'help': 'Number of steps between validation runs. Epochs in this case'}}},
             'checkpointing': {'save_top_k': {'kwargs': {'help': 'random comment Save top K model checkpoints during training.'}},
              'checkpoint_every': {'kwargs': {'help': 'Number of epochs between checkpoints'}}},
             'dataloaders': {'persistent_workers': {'kwargs': {'help': 'Workers for the dataloader'}}}})

In [5]:
def update_nested_dict(o_dict, u_dict):
    # nested_key_path = []
    for u_key, u_item in u_dict.items():
        if isinstance(u_item, dict) and len(u_item) > 0:
            update_nested_dict(o_dict[u_key], u_item)
        else: 
            # print(u_key, u_item)
            # print('out', o_dict)
            o_dict[u_key] = u_item
            # print(nested_key_path)
    return o_dict

# print(dict(update_dict))
updated_dict = update_nested_dict(dict(t_config), dict(update_dict))

In [6]:
from collections.abc import MutableMapping

def flatten(dictionary, parent_key='', separator='_'):
    items = []
    for key, value in dictionary.items():
        new_key = parent_key + separator + key if parent_key else key
        if isinstance(value, MutableMapping):
            if sorted(list(value.keys())) == ['args','kwargs']:
                if len(value['args']) == 0:
                    value['args'] = [f'--{key}', f'-{key[0]}']
                items.append((new_key, value))
            else:
                items.extend(flatten(value, new_key, separator=separator).items())
        else:
            items.append((new_key, value))
    return dict(items)

In [7]:
f_dict =flatten(updated_dict, separator='.')
f_dict

{'logging.name': {'args': ['--name', '-n'],
  'kwargs': {'default': 'Example Run',
   'help': 'Name of the run. Used for logging.'}},
 'logging.project': {'args': ['--project', '-p'],
  'kwargs': {'default': 'Example Project',
   'help': 'Name of the project. Used for logging.'}},
 'logging.wandb.run_id': {'args': ['--project', '-p'],
  'kwargs': {'default': 'Example Run',
   'help': 'You can specify a run-id if you would like wandb to continue logging from that checkpoint.'}},
 'logging.wandb.ckpt_name': {'args': ['--ckpt_name', '-c'],
  'kwargs': {'default': 'example_ckpt'}},
 'training.seed': {'args': ['--seed', '-s'],
  'kwargs': {'default': 42, 'help': 'The random seed'}},
 'training.batch_size': {'args': ['--batch_size', '-b'],
  'kwargs': {'default': 256, 'help': 'The batch size'}},
 'training.precision': {'args': ['--precision', '-p'],
  'kwargs': {'default': '16-mixed', 'help': 'Precision to use for training'}},
 'training.strategy': {'args': ['--strategy', '-s'],
  'kwargs': 

In [73]:
f_dict

{'logging.name': {'args': ['--name', '-n'],
  'kwargs': {'default': 'Example Run',
   'help': 'Name of the run. Used for logging.'}},
 'logging.project': {'args': ['--project', '-p'],
  'kwargs': {'default': 'Example Project',
   'help': 'Name of the project. Used for logging.'}},
 'logging.wandb.run_id': {'args': ['--project', '-p'],
  'kwargs': {'default': 'Example Run',
   'help': 'You can specify a run-id if you would like wandb to continue logging from that checkpoint.'}},
 'logging.wandb.ckpt_name': {'args': ['--ckpt_name', '-c'],
  'kwargs': {'default': 'example_ckpt'}},
 'training.seed': {'args': ['--seed', '-s'],
  'kwargs': {'default': 42, 'help': 'The random seed'}},
 'training.batch_size': {'args': ['--batch_size', '-b'],
  'kwargs': {'default': 256, 'help': 'The batch size'}},
 'training.precision': {'args': ['--precision', '-p'],
  'kwargs': {'default': '16-mixed', 'help': 'Precision to use for training'}},
 'training.strategy': {'args': ['--strategy', '-s'],
  'kwargs': 

In [98]:
from copy import deepcopy
names_set = set()
flags_set = set()
final_dict = deepcopy(f_dict)

for key, val in final_dict.items():
    # print(key)
    args = val['args']
    
    key_split = list(reversed(key.split('.')))

    meta_var = None
    if len(args) == 1:
        k_depth_idx = 1
        n_c = args[0]
        while n_c in names_set:
            n_c = key_split[k_depth_idx] + "_" + n_c
            k_depth_idx += 1
        names_set.add(n_c)
        final_dict[key]['args'] = [n_c]


    elif len(args) > 1:
        out_args = []
        for a in sorted(args, reverse=True):
            f_c = a
            c_idx = 0
            k_depth_idx = 0

            if f_c[0:2] == '--':
                meta_var = f_c[2:]

            while f_c in flags_set:
                if f_c[0:2] == '--':
                    f_c = '--' + key_split[k_depth_idx  + 1] + "_" + f_c[2:]
                    meta_var = f_c[2:]
                    k_depth_idx += 1
                elif f_c[0:1] == '-':
                    if k_depth_idx >= len(key_split):
                        import random
                        import string
                        
                        # Get all lowercase letters
                        lowers = set(
                               [f'-{l}' for l in string.ascii_lowercase]
                        )
                        uppers = set(
                                [f'-{l}' for l in string.ascii_uppercase]
                        )

                        # Find letters that are not in the excluded set
                        available_lowers = list(lowers - flags_set)
                        available_uppers = list(uppers - flags_set)
                        
                        if len(available_lowers) > 0:
                            f_c = random.choice(available_lowers)
                        elif len(available_uppers) > 0:
                            f_c = random.choice(available_uppers)

                    else:    
                        if c_idx >= len(key_split[k_depth_idx]):
                            k_depth_idx += 1
                            c_idx = 0

                        f_c = '-' + key_split[k_depth_idx][c_idx]
                        if f_c in flags_set:
                            f_c = '-' + key_split[k_depth_idx][c_idx].capitalize()

                        c_idx += 1
                else:
                    pass

            c_idx = 0
            k_depth_idx = 0
            flags_set.add(f_c)
            out_args.append(f_c)
        
        final_dict[key]['args'] = out_args
        if 'metavar' not in final_dict[key]['kwargs']:
            final_dict[key]['kwargs']['metavar'] = meta_var
    else:
        pass
    
final_dict

{'logging.name': {'args': ['-n', '--name'],
  'kwargs': {'default': 'Example Run',
   'help': 'Name of the run. Used for logging.',
   'metavar': 'name'}},
 'logging.project': {'args': ['-p', '--project'],
  'kwargs': {'default': 'Example Project',
   'help': 'Name of the project. Used for logging.',
   'metavar': 'project'}},
 'logging.wandb.run_id': {'args': ['-r', '--wandb_project'],
  'kwargs': {'default': 'Example Run',
   'help': 'You can specify a run-id if you would like wandb to continue logging from that checkpoint.',
   'metavar': 'wandb_project'}},
 'logging.wandb.ckpt_name': {'args': ['-c', '--ckpt_name'],
  'kwargs': {'default': 'example_ckpt', 'metavar': 'ckpt_name'}},
 'training.seed': {'args': ['-s', '--seed'],
  'kwargs': {'default': 42, 'help': 'The random seed', 'metavar': 'seed'}},
 'training.batch_size': {'args': ['-b', '--batch_size'],
  'kwargs': {'default': 256,
   'help': 'The batch size',
   'metavar': 'batch_size'}},
 'training.precision': {'args': ['-P', '-

In [99]:
import argparse

In [100]:
p = argparse.ArgumentParser()

In [101]:
for v in final_dict.values():
    p.add_argument(*v['args'], **v['kwargs'])

In [102]:
p.print_help()

usage: ipykernel_launcher.py [-h] [-n name] [-p project] [-r wandb_project]
                             [-c ckpt_name] [-s seed] [-b batch_size]
                             [-P precision] [-S strategy] [-a save_dir]
                             [-v val_every] [-A save_top_k]
                             [-C checkpoint_every] [-N num_workers]
                             [-e persistent_workers] [-R recover_latest]
                             [-k ckpt_path] [-K pretrained_ckpt_path]

options:
  -h, --help            show this help message and exit
  -n name, --name name  Name of the run. Used for logging.
  -p project, --project project
                        Name of the project. Used for logging.
  -r wandb_project, --wandb_project wandb_project
                        You can specify a run-id if you would like wandb to
                        continue logging from that checkpoint.
  -c ckpt_name, --ckpt_name ckpt_name
  -s seed, --seed seed  The random seed
  -b batch_size, --batch

In [103]:
parsed = p.parse_args([])
parsed

Namespace(name='Example Run', project='Example Project', wandb_project='Example Run', ckpt_name='example_ckpt', seed=42, batch_size=256, precision='16-mixed', strategy='auto', save_dir='/home/eduardo/Projects/pre_encode_audio/data/hyperencoder_training', val_every=-1, save_top_k=10, checkpoint_every=1, num_workers=8, persistent_workers=False, recover_latest=False, ckpt_path='', pretrained_ckpt_path='')

In [105]:
reverse_meta_mapping = {}
for key, vals in final_dict.items():
    reverse_meta_mapping[vals['kwargs']['metavar']] = key.split('.')

reverse_meta_mapping

{'name': ['logging', 'name'],
 'project': ['logging', 'project'],
 'wandb_project': ['logging', 'wandb', 'run_id'],
 'ckpt_name': ['logging', 'wandb', 'ckpt_name'],
 'seed': ['training', 'seed'],
 'batch_size': ['training', 'batch_size'],
 'precision': ['training', 'precision'],
 'strategy': ['training', 'strategy'],
 'save_dir': ['training', 'save_dir'],
 'val_every': ['validation', 'val_every'],
 'save_top_k': ['checkpointing', 'save_top_k'],
 'checkpoint_every': ['checkpointing', 'checkpoint_every'],
 'num_workers': ['dataloaders', 'num_workers'],
 'persistent_workers': ['dataloaders', 'persistent_workers'],
 'recover_latest': ['recovery', 'recover_latest'],
 'ckpt_path': ['recovery', 'ckpt_path'],
 'pretrained_ckpt_path': ['pretrained', 'ckpt_path']}

In [108]:
for v in vars(parsed):
    print(getattr(parsed, v))

Example Run
Example Project
Example Run
example_ckpt
42
256
16-mixed
auto
/home/eduardo/Projects/pre_encode_audio/data/hyperencoder_training
-1
10
1
8
False
False




In [175]:
def namespace_builder(init_namespace, namespace_mapping):
        base_namespace = argparse.Namespace()
        # bootstrap obj architecture
        for k, v in namespace_mapping.items():
            # print(k, v)
            # print("Current State:", vars(self))
            curr_attr = base_namespace
            n_l = len(v)
            for idx, l in enumerate(v):
                if idx == n_l - 1:
                    # print(f"Setting {k} to None")
                    setattr(curr_attr, l, None)
                else:
                    if isinstance(getattr(curr_attr, l, None), argparse.Namespace):
                        curr_attr = getattr(curr_attr, l)
                    else:
                        setattr(curr_attr, l, argparse.Namespace())
                        curr_attr = getattr(curr_attr, l)
                idx+=1
        # Insert vars in original namespace into bootstrapped namespace
        # based on the reveresed key mapping.
        for v in vars(init_namespace):
            curr_attr = base_namespace
            key_path = namespace_mapping[v]
            for i, k in enumerate(key_path):
                if i == len(key_path) - 1:
                    setattr(curr_attr, k, getattr(init_namespace, v))
                else:
                    curr_attr = getattr(curr_attr, k)
        
        return base_namespace

            

args = namespace_builder(parsed, reverse_meta_mapping)

In [177]:
args

Namespace(logging=Namespace(name='Example Run', project='Example Project', wandb=Namespace(run_id='Example Run', ckpt_name='example_ckpt')), training=Namespace(seed=42, batch_size=256, precision='16-mixed', strategy='auto', save_dir='/home/eduardo/Projects/pre_encode_audio/data/hyperencoder_training'), validation=Namespace(val_every=-1), checkpointing=Namespace(save_top_k=10, checkpoint_every=1), dataloaders=Namespace(num_workers=8, persistent_workers=False), recovery=Namespace(recover_latest=False, ckpt_path=''), pretrained=Namespace(ckpt_path=''))